# Understanding LLM Limits & Prompting Techniques

This notebook covers two foundational ideas about working with hosted LLMs like `gpt-4o-mini`:

1. **The Knowledge Cutoff** — why a plain model call can't tell you today's date (or anything after its training data ends).
2. **Five Prompting Techniques** — Chain of Thought, Few-Shot, Structured Output, and Prompt Chaining: minimal runnable demos of each.

---
## Step 1: Install Dependencies

In [ ]:
!pip install -q openai python-dotenv

---
## Step 2: Set Up Environment Variables & Client
Securely load your OpenAI API key and instantiate the OpenAI client.

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

client = OpenAI()
MODEL = "gpt-4o-mini"

def ask(messages, **kwargs):
    return client.chat.completions.create(model=MODEL, messages=messages, **kwargs).choices[0].message.content

---
## Step 3: The Knowledge Cutoff
With no tools, no web search, and no system date injected, a plain model call can only draw on what it learned during training. It has no clock and no live connection to the outside world, so it cannot know today's date.

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "what is today's date."}],
)

print(response.choices[0].message.content)

---
## Step 4: Chain of Thought
Asking the model to reason step by step before answering improves accuracy on problems that need intermediate steps (arithmetic, logic) instead of a single leap to the answer.

The trap in the question below: the afternoon discount applies to the *remaining* apples, not the original total. Asked directly, the model often skips that nuance and answers wrong (30 or 36); asked to reason step by step, it reliably arrives at the correct answer, 45.

In [ ]:
def cot_demo():
    question = "A store had 96 apples. They sold 3/8 of them in the morning and 1/4 of the remaining apples in the afternoon. How many apples are left?"

    direct = ask([{"role": "user", "content": question + " Answer with just the number."}])

    cot = ask([{"role": "user", "content": question + " Let's think step by step."}])

    print(f"Direct answer: {direct}")
    print(f"Chain-of-thought answer:
{cot}")

cot_demo()

---
## Step 5: Few-Shot Prompting
Show the model a handful of input → output examples in the prompt so it infers the task and output format, instead of relying on an instruction alone.

In [ ]:
FEW_SHOT_EXAMPLES = [
    {"role": "user", "content": "This laptop's battery dies in two hours."},
    {"role": "assistant", "content": "negative"},
    {"role": "user", "content": "The delivery arrived a day early!"},
    {"role": "assistant", "content": "positive"},
    {"role": "user", "content": "The manual is 40 pages long."},
    {"role": "assistant", "content": "neutral"},
]

def few_shot_demo(text="Support replied within five minutes and fixed everything."):
    messages = [
        {"role": "system", "content": "Classify the sentiment of each message as positive, negative, or neutral. Reply with one word."},
        *FEW_SHOT_EXAMPLES,
        {"role": "user", "content": text},
    ]
    print(f"'{text}' -> {ask(messages)}")

few_shot_demo()

---
## Step 6: Structured Output
Force the model's response into a fixed JSON schema so it can be parsed and used programmatically, instead of free-form text.

In [ ]:
import json

EVENT_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "event",
        "schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "date": {"type": "string"},
                "city": {"type": "string"},
            },
            "required": ["title", "date", "city"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

def structured_demo():
    text = "Rootconf will be held in Bangalore on 14 March 2026."
    reply = ask(
        [
            {"role": "system", "content": "Extract the event details from the text."},
            {"role": "user", "content": text},
        ],
        response_format=EVENT_SCHEMA,
    )
    print(json.dumps(json.loads(reply), indent=2))

structured_demo()

---
## Step 7: Prompt Chaining
Break a task into steps and feed each step's output into the next prompt, instead of asking for everything in one shot.

In [ ]:
def chaining_demo():
    topic = "why coffee tastes bitter"

    draft = ask([{"role": "user", "content": f"Write a two-sentence explanation of {topic}."}])
    print(f"Draft:\n{draft}\n")

    simplified = ask([{"role": "user", "content": f"Rewrite this so a 7-year-old understands it:\n{draft}"}])
    print(f"Simplified:\n{simplified}")

chaining_demo()